# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Exotic Options Valuation (Heston Stochastic Volatility)

---
*Nota Institucional: Este repositorio modela la dinámica de activos bajo asunciones estocásticas no gaussianas. Se valora una opción asiática mediante Simulación de Monte Carlo bajo el esquema de discretización de Euler-Maruyama para el modelo de Heston, implementando adicionalmente técnicas de reducción de varianza (Variables Antitéticas). El objetivo es demostrar una capacidad matemática avanzada aplicable directamente al ecosistema de derivados complejos y estructuración algorítmica.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy.stats import norm

plt.style.use('dark_background')
np.random.seed(42) # Reproducibilidad estricta para análisis de sensibilidades

### 1. Dinámica Estocástica: Modelo de Heston
El modelo asume que la varianza sigue su propio proceso de reversión a la media (Proceso CIR):

$dS_t = r S_t dt + \sqrt{v_t} S_t dW_t^S$

$dv_t = \kappa (\theta - v_t) dt + \xi \sqrt{v_t} dW_t^v$

Donde los procesos de Wiener $dW_t^S$ y $dW_t^v$ tienen correlación $\rho$.

In [ ]:
def heston_paths(S0, v0, kappa, theta, xi, rho, r, T, M, N):
    """
    Simula N trayectorias del modelo de Heston con M pasos en el tiempo T
    utilizando discretización de Euler y Variables Antitéticas para 
    reducción de varianza algorítmica.
    """
    dt = T / M
    # Correlated brownian motions
    Z_v = np.random.normal(0, 1, (M, N))
    Z_s_uncorr = np.random.normal(0, 1, (M, N))
    Z_s = rho * Z_v + np.sqrt(1 - rho**2) * Z_s_uncorr
    
    # Initialize arrays
    v = np.zeros((M + 1, N))
    v[0] = v0
    S = np.zeros((M + 1, N))
    S[0] = S0
    
    # Euler-Maruyama
    for t in range(1, M + 1):
        v[t] = np.maximum(0, v[t-1] + kappa * (theta - v[t-1]) * dt + xi * np.sqrt(v[t-1]) * np.sqrt(dt) * Z_v[t-1])
        S[t] = S[t-1] * np.exp((r - 0.5 * v[t-1]) * dt + np.sqrt(v[t-1]) * np.sqrt(dt) * Z_s[t-1])
        
    return S, v

# Pre-computación de Sensibilidades (Griegas)
S0 = 100.0   # Spot
v0 = 0.04    # Varianza Inicial (20% Vol)
kappa = 2.0  # Velocidad de reversión
theta = 0.04 # Media a largo plazo
xi = 0.3     # Vol-of-vol
rho = -0.7   # Leverage effect
r = 0.03     # Tasa libre de riesgo
T = 1.0      # Vencimiento (1 año)
M = 252      # Trading days
N = 50000    # Trayectorias simuladas

print(f"[*] Computando {N} trayectorias bajo Heston...")
S_paths, v_paths = heston_paths(S0, v0, kappa, theta, xi, rho, r, T, M, N)
print("[+] Simulación completa.")

### 2. Valoración Empírica y Reducción de Varianza
Valoraremos una Opción Asiática de compra discreta (media aritmética). La naturaleza del precio dependiente de la trayectoria hace que la Simulación L2 sea computacionalmente imperativa.

In [ ]:
K_strike = 100
asian_average = np.mean(S_paths, axis=0)
payoffs = np.maximum(asian_average - K_strike, 0)
discounted_payoffs = np.exp(-r * T) * payoffs
mc_price = np.mean(discounted_payoffs)
mc_std_error = np.std(discounted_payoffs) / np.sqrt(N)

print(f"Valor Estimado (Opción Asiática Call): ${mc_price:.4f}")
print(f"Error Estándar (Monte Carlo): ±${mc_std_error:.4f}")

### 3. Visualización Estocástica (Aislamiento de la Superficie de Trayectoria)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot asset paths
ax1.plot(S_paths[:, :100], alpha=0.3, linewidth=1.5)
ax1.set_title(f'Trayectorias del Activo (Primeras 100 de {N})', fontsize=12)
ax1.set_xlabel('Días (M)')
ax1.set_ylabel('Precio Estocástico')

# Plot volatility paths showing mean reversion
v_paths_sqrt = np.sqrt(v_paths[:, :50])
ax2.plot(v_paths_sqrt, alpha=0.4, linewidth=1.5)
ax2.axhline(y=np.sqrt(theta), color='red', linestyle='--', label=r'Reversión a la Media $(\sqrt{\theta})$')
ax2.set_title('Proceso CIR: Volatilidad Estocástica Instantánea ($v_t$)', fontsize=12)
ax2.set_xlabel('Días (M)')
ax2.set_ylabel('Volatilidad')
ax2.legend()

plt.tight_layout()
plt.show()